<a href="https://colab.research.google.com/github/aliza1800/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliza1800/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I chose Random Forest because the task is to identify pages that are likely to be declining. Random Forest can capture non-linear relationships between the available signals and the declining label, while also providing feature importance for interpretation. It is a reasonable next model to compare against the Week-4 baseline because the goal is not to reward complexity, but to test whether a more flexible model improves the same evaluation metric on the same data.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I use a stratified train-test split so that the proportion of declining and non-declining pages is similar in both sets. I use the same split design and evaluation metric as the Week-4 baseline so the model comparison is fair. The test set is kept separate from training and is used only for final evaluation.


In [13]:
import os

print(os.path.exists("/content/flyrank-ml-internship"))
print(os.listdir("/content")[:20])

True
['.config', 'flyrank-ml-internship', 'sample_data']


In [14]:
!git clone https://github.com/aliza1800/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [15]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

print("Dataset loaded:", df.shape)

Dataset loaded: (30000, 44)


In [16]:
# Define the target
df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

# Use the same pre-decision features as the Week-4 baseline
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df["is_declining_label"]

print("Features:", features)
print("Rows:", len(X))
print("Declining rate:", round(y.mean(), 3))

Features: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']
Rows: 30000
Declining rate: 0.542


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train declining rate:", round(y_train.mean(), 3))
print("Test declining rate:", round(y_test.mean(), 3))

Training rows: 24000
Test rows: 6000
Train declining rate: 0.542
Test declining rate: 0.542


In [18]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [19]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Predict probability of the "declining" class on the test set
rf_score = model.predict_proba(X_test)[:, 1]

print("Random Forest evaluation")
print("-" * 30)

for k in (20, 50):
    print(
        f"Precision@{k}: "
        f"{precision_at_k(rf_score, y_test, k):.3f}"
    )

Random Forest evaluation
------------------------------
Precision@20: 0.900
Precision@50: 0.920


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Model comparison

On the same held-out test set, the Random Forest outperformed the Week-4 hand-rule baseline on both evaluation metrics. Precision@20 was 0.900 for the Random Forest versus 0.650 for the hand rule, while Precision@50 was 0.920 versus 0.640. This indicates that the Random Forest provided stronger ranking performance on this test set, although the result should be treated as decision-support evidence rather than proof that the more complex model will always perform better.


In [20]:
# Week-4 hand-rule baseline on the same test set
stale = (X_test["days_since_last_update"] >= 180).astype(int)
visible = (X_test["impressions_90d"] >= 500).astype(int)

hand_rule_score = (
    stale * visible * X_test["impressions_90d"]
)

# Compare Random Forest with the hand-rule baseline
comparison = []

for k in (20, 50):
    comparison.append({
        "Metric": f"Precision@{k}",
        "Week-4 Hand Rule": precision_at_k(hand_rule_score, y_test, k),
        "Random Forest": precision_at_k(rf_score, y_test, k)
    })

comparison_df = pd.DataFrame(comparison)

print(comparison_df.to_string(index=False))


      Metric  Week-4 Hand Rule  Random Forest
Precision@20              0.65           0.90
Precision@50              0.64           0.92


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

The Random Forest produced 1,082 false positives and 816 false negatives at the default 0.5 classification threshold. This shows that the model still makes both types of errors, so its predictions should be treated as decision-support signals rather than perfect classifications.

The most important features were `impressions_90d` (0.268), `avg_position` (0.250), `content_age_days` (0.163), and `word_count` (0.162). `ctr` contributed 0.114, while `days_since_last_update` had the lowest importance at 0.043. Overall, the model appears to rely more on current search visibility and ranking signals than on update recency alone.


In [21]:
from sklearn.metrics import confusion_matrix

# Classification errors at the default 0.5 threshold
y_pred = model.predict(X_test)

false_positives = ((y_test == 0) & (y_pred == 1)).sum()
false_negatives = ((y_test == 1) & (y_pred == 0)).sum()

print("Error analysis")
print("-" * 30)
print("False positives:", false_positives)
print("False negatives:", false_negatives)

# Feature importance
importance = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

print("\nFeature importance")
print(importance.round(3))

Error analysis
------------------------------
False positives: 1082
False negatives: 816

Feature importance
impressions_90d           0.268
avg_position              0.250
content_age_days          0.163
word_count                0.162
ctr                       0.114
days_since_last_update    0.043
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.